# Remote Coding Agent: one-click Colab deployment

Run the deployment cell once. It clones/updates the repository, installs dependencies, verifies the Cloudflare Quick Tunnel package, starts the authenticated MCP server, checks the public endpoint, and prints the connection fields.

**Security:** this tool executes shell commands. The temporary tunnel is public, so authentication stays enabled. Do not put production credentials or sensitive source code in a disposable Colab runtime. The bearer token is entered through hidden input and is never printed.

In [ ]:
# One-click deployment: run this cell once, then copy the printed MCP URL into your client.
import hashlib
import json
import os
import pathlib
import platform
import re
import signal
import subprocess
import time
import urllib.request
from getpass import getpass

import httpx

REPO_URL = "https://github.com/Dishant402955/remote-coding-agent.git"
REPO_DIR = pathlib.Path("/content/remote-coding-agent")
WORKSPACE = pathlib.Path("/content/remote-coding-agent-workspace")
SERVER_LOG = pathlib.Path("/content/remote-coding-agent-server.log")
TUNNEL_LOG = pathlib.Path("/content/remote-coding-agent-cloudflared.log")

SERVER_NAME = "Remote Coding Agent"
SERVER_DESCRIPTION = (
    "Run development commands in a remote workspace using "
    "the run_command MCP tool."
)
CONNECTION_TYPE = "Remote MCP / Streamable HTTP"
TUNNEL_PROVIDER = "Cloudflare Quick Tunnel (temporary HTTPS)"
AUTH_MODE = "bearer"

MCP_AUTH_TOKEN = getpass(
    "Create/paste a fresh MCP bearer token (hidden input, 32+ chars): "
).strip()
if len(MCP_AUTH_TOKEN) < 32:
    raise ValueError("Use a high-entropy token of at least 32 characters. Input is hidden.")

def stop_process(process):
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 5
    while time.monotonic() < deadline:
        try:
            os.killpg(process.pid, 0)
        except ProcessLookupError:
            break
        time.sleep(0.1)
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        pass

# Stop any previous run of this cell.
stop_process(globals().get("tunnel"))
stop_process(globals().get("server"))

# Clone or fast-forward the public repository.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True, timeout=120)
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository; inspect it before retrying."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
        timeout=120,
    )

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-e", ".", "httpx"],
    cwd=REPO_DIR,
    check=True,
    timeout=180,
)
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Download cloudflared only after verifying the published SHA-256 digest.
asset_names = {
    "x86_64": "cloudflared-linux-amd64.deb",
    "amd64": "cloudflared-linux-amd64.deb",
    "aarch64": "cloudflared-linux-arm64.deb",
    "arm64": "cloudflared-linux-arm64.deb",
}
asset_name = asset_names.get(platform.machine().lower())
if not asset_name:
    raise RuntimeError(f"Unsupported Colab architecture: {platform.machine()}")

request = urllib.request.Request(
    "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
    headers={"Accept": "application/vnd.github+json"},
)
with urllib.request.urlopen(request, timeout=20) as response:
    release = json.load(response)
asset = next((item for item in release["assets"] if item["name"] == asset_name), None)
if asset is None or not (asset.get("digest") or "").startswith("sha256:"):
    raise RuntimeError("Latest cloudflared release has no verifiable SHA-256 asset digest.")

deb_path = pathlib.Path("/content/cloudflared.deb")
hasher = hashlib.sha256()
with urllib.request.urlopen(asset["browser_download_url"], timeout=30) as response:
    with deb_path.open("wb") as destination:
        total_bytes = 0
        while chunk := response.read(1024 * 1024):
            total_bytes += len(chunk)
            if total_bytes > 200 * 1024 * 1024:
                raise RuntimeError("cloudflared package exceeds 200 MiB.")
            hasher.update(chunk)
            destination.write(chunk)
if f"sha256:{hasher.hexdigest()}" != asset["digest"]:
    deb_path.unlink(missing_ok=True)
    raise RuntimeError("cloudflared SHA-256 verification failed.")
subprocess.run(["dpkg", "-i", str(deb_path)], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["cloudflared", "--version"], check=True, stdout=subprocess.DEVNULL)

# Pass only the required MCP credential to the server process.
server_env = os.environ.copy()
for key in list(server_env):
    secret_markers = ("TOKEN", "SECRET", "PASSWORD", "API_KEY", "CREDENTIAL")
    if any(word in key.upper() for word in secret_markers):
        server_env.pop(key, None)
server_env.update({
    "MCP_AUTH_TOKEN": MCP_AUTH_TOKEN,
    "REMOTE_CODING_AGENT_AUTH_MODE": AUTH_MODE,
    "REMOTE_CODING_AGENT_HOST": "127.0.0.1",
    "REMOTE_CODING_AGENT_PORT": "8000",
    "REMOTE_CODING_AGENT_WORKSPACE": str(WORKSPACE),
})
server_log_handle = SERVER_LOG.open("w", encoding="utf-8")
server = subprocess.Popen(
    ["remote-coding-agent"],
    cwd=REPO_DIR,
    env=server_env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

deadline = time.monotonic() + 20
while time.monotonic() < deadline:
    if server.poll() is not None:
        server_log_handle.close()
        raise RuntimeError(
            f"Server exited. Inspect {SERVER_LOG} locally; "
            "logs may contain sensitive output."
        )
    try:
        probe = httpx.get("http://127.0.0.1:8000/", timeout=1)
        if probe.status_code in (200, 401, 404, 405):
            break
    except Exception:
        time.sleep(0.25)
else:
    stop_process(server)
    server_log_handle.close()
    raise RuntimeError("MCP server did not start within 20 seconds.")

tunnel_log_handle = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
tunnel_url = None
deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    if tunnel.poll() is not None:
        break
    try:
        log_text = TUNNEL_LOG.read_text(encoding="utf-8", errors="replace")
    except OSError:
        log_text = ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.25)
tunnel_log_handle.close()
server_log_handle.close()
if not tunnel_url:
    stop_process(tunnel)
    stop_process(server)
    raise RuntimeError(f"Cloudflare URL was not detected. Inspect {TUNNEL_LOG} locally.")

MCP_ENDPOINT = tunnel_url + "/mcp"
HEADERS = {
    "Authorization": f"Bearer {MCP_AUTH_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
}
initialize_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "colab-one-click-deployment", "version": "1.0"},
    },
}
local_response = httpx.post(
    "http://127.0.0.1:8000/mcp", headers=HEADERS, json=initialize_payload, timeout=15
)
if local_response.status_code != 200:
    stop_process(tunnel)
    stop_process(server)
    raise RuntimeError(f"Local MCP initialize failed: HTTP {local_response.status_code}")
public_response = httpx.post(MCP_ENDPOINT, headers=HEADERS, json=initialize_payload, timeout=25)
if public_response.status_code != 200:
    stop_process(tunnel)
    stop_process(server)
    raise RuntimeError(f"Public tunnel MCP initialize failed: HTTP {public_response.status_code}")

print("\n=== ChatGPT MCP connection details ===")
print(f"Name: {SERVER_NAME}")
print(f"Description: {SERVER_DESCRIPTION}")
print(f"Connection: {CONNECTION_TYPE}")
print("Tunnel: cloudflare")
print(f"Tunnel type: {TUNNEL_PROVIDER}")
print(f"Tunnel base URL: {tunnel_url}")
print(f"MCP server URL: {MCP_ENDPOINT}")
print("Authentication: Bearer token")
print(
    "Authentication token: kept hidden; use the same value you entered above "
    "if your client supports bearer credentials."
)
print("\nStatus: local and public MCP initialization checks passed.")
print(
    "Important: this tunnel is public. Keep bearer authentication enabled; "
    "do not switch this shell runner to no-auth mode."
)
print("Keep this Colab runtime running while connected. The URL changes when the tunnel restarts.")

## Stop the server and tunnel

Run the following in a new code cell before ending the runtime:

```python
stop_process(globals().get("tunnel"))
stop_process(globals().get("server"))
print("MCP server and Cloudflare tunnel stopped.")
```

The Quick Tunnel URL is temporary. The ChatGPT connection must use the full MCP server URL ending in `/mcp`, not the base tunnel URL.